# Structured Streaming on Databricks

> **Prerequisites:** This notebook assumes you've read the Structured Streaming notebooks in the `apache-spark` repo (15–17), which cover the unbounded table model, output modes, triggers, watermarking, stateful aggregations, `foreachBatch` basics, and stream-static joins. This notebook focuses on the **Databricks-specific patterns**: Delta as a streaming source, `startingVersion`/`startingTimestamp`, the CDF streaming source, `foreachBatch` + MERGE, `.toTable()`, and query management.

In this notebook we'll cover:
- Reading from a Delta table as a stream
- Controlling where the stream starts with `startingVersion` and `startingTimestamp`
- Change Data Feed (CDF) as a streaming source
- `foreachBatch` + `MERGE` — the exam-critical pattern for streaming upserts
- `.toTable()` vs `.start(path)` for Delta sinks
- Managing and monitoring active streaming queries

## Delta as a Streaming Source

Any Delta table can be read as a stream. Spark tracks which Delta table versions have been processed (via the stream checkpoint) and delivers only new data — rows added in new commits — on each micro-batch.

```python
# Read a Delta path as a stream
stream = spark.readStream \
              .format("delta") \
              .load("s3://bucket/bronze/orders/")

# Or read a registered Delta table by name
stream = spark.readStream.table("bronze.orders")
```

Every time a new commit lands on the source Delta table (from a batch write, another stream, Auto Loader, or MERGE), the streaming reader picks up the new rows in the next micro-batch.

### What the Delta Streaming Source Delivers

- **Append-only by default** — new rows from `INSERT` or `INSERT OVERWRITE` operations
- **Does not deliver deletes or updates** by default — those require Change Data Feed (below)
- Reads are **consistent** — each micro-batch is a complete Delta snapshot version, never a partial write

### Ignoring Updates and Deletes

If the source Delta table has updates or deletes, the stream will fail by default (non-append data cannot be delivered in append output mode). Use `ignoreChanges` to skip changed files:

```python
spark.readStream \
     .format("delta") \
     .option("ignoreChanges", "true") \
     .table("bronze.orders")
```

> `ignoreChanges` means updated rows may be re-delivered — the stream re-reads the whole rewritten file. Use it only when downstream processing is idempotent. For true change delivery, use CDF.

## Starting Position — `startingVersion` and `startingTimestamp`

By default, a new Delta stream starts at the **latest version** of the table (processes only future commits). To replay historical data, override the starting position:

```python
# Start from a specific Delta table version
spark.readStream \
     .format("delta") \
     .option("startingVersion", 5) \
     .table("bronze.orders")

# Start from the beginning of all recorded history
spark.readStream \
     .format("delta") \
     .option("startingVersion", "earliest") \
     .table("bronze.orders")

# Start from a specific point in time
spark.readStream \
     .format("delta") \
     .option("startingTimestamp", "2024-01-15T09:00:00") \
     .table("bronze.orders")
```

> `startingVersion` and `startingTimestamp` only apply when the stream starts **without an existing checkpoint**. If a checkpoint exists, the stream resumes from where it left off — these options are ignored.

## Change Data Feed (CDF) as a Streaming Source

The standard Delta streaming source delivers only appended rows. **Change Data Feed** delivers every change — inserts, updates, and deletes — as individual change records with an operation type column.

Enable CDF on the source table:

```sql
ALTER TABLE silver.customers
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');
```

Read CDF as a stream:

```python
cdf_stream = (
    spark.readStream
         .format("delta")
         .option("readChangeFeed",    "true")
         .option("startingVersion",   1)        # version when CDF was enabled
         .table("silver.customers")
)
```

### CDF Record Schema

CDF adds three system columns to every record:

| Column | Type | Description |
|--------|------|-------------|
| `_change_type` | STRING | `insert`, `update_preimage`, `update_postimage`, `delete` |
| `_commit_version` | LONG | Delta version of the commit that produced this change |
| `_commit_timestamp` | TIMESTAMP | When the commit happened |

For an `UPDATE`, CDF emits **two rows**: `update_preimage` (the old values) and `update_postimage` (the new values). For `INSERT` and `DELETE`, it emits one row each.

### CDF Use Case: Propagating Changes Downstream

```python
# Read only update_postimage and insert records → apply to a Gold table
from pyspark.sql.functions import col

(
    spark.readStream
         .format("delta")
         .option("readChangeFeed",  "true")
         .option("startingVersion", 1)
         .table("silver.customers")
         .filter(col("_change_type").isin("insert", "update_postimage"))
         .drop("_change_type", "_commit_version", "_commit_timestamp")
         .writeStream
         .format("delta")
         .option("checkpointLocation", "/ckpt/gold/customer_summary")
         .outputMode("append")
         .trigger(availableNow=True)
         .toTable("gold.customer_summary")
)

## `foreachBatch` + MERGE — Streaming Upserts

The most exam-tested streaming pattern on Databricks. `foreachBatch` receives each micro-batch as a static DataFrame and lets you run arbitrary logic — including `MERGE INTO` — against a Delta table.

This is the standard approach when:
- The stream contains updates and deletes (not just appends)
- You need to write to multiple targets from one stream
- You need to apply transformations that aren't expressible in `writeStream` directly

```python
def upsert_to_silver(batch_df, batch_id):
    """Called once per micro-batch with the batch as a static DataFrame."""
    # Deduplicate within the batch (multiple changes per key)
    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number, desc

    w = Window.partitionBy("cust_id").orderBy(desc("updated_at"))
    deduped = batch_df.withColumn("rn", row_number().over(w)) \
                      .filter("rn = 1").drop("rn")

    # Register as a temp view for SQL MERGE
    deduped.createOrReplaceTempView("stream_batch")

    # MERGE the batch into the Silver table
    batch_df.sparkSession.sql("""
        MERGE INTO silver.customers AS t
        USING stream_batch           AS s
        ON    t.cust_id = s.cust_id

        WHEN MATCHED AND s.op = 'D' THEN DELETE
        WHEN MATCHED               THEN UPDATE SET *
        WHEN NOT MATCHED           THEN INSERT *
    """)

(
    spark.readStream
         .table("bronze.cdc_orders")
         .writeStream
         .foreachBatch(upsert_to_silver)
         .option("checkpointLocation", "/ckpt/silver/customers")
         .trigger(availableNow=True)
         .start()
)
```

**Key points:**
- `batch_df` is a **static** DataFrame — all static DataFrame operations and SQL work on it
- Use `batch_df.sparkSession.sql(...)` (not the outer `spark`) to ensure the SQL runs in the correct session context
- The `batch_id` parameter is an incrementing integer — useful for idempotency checks
- `foreachBatch` with `MERGE` gives **exactly-once** upsert semantics when combined with a checkpoint

> **Exam tip:** `foreachBatch` does **not** use `outputMode` — it's ignored. The write logic inside `foreachBatch` controls what happens to the data.

## `.toTable()` vs `.start(path)`

Two ways to sink a stream to Delta:

```python
# Option A — write to a path (unregistered)
.writeStream \
 .format("delta") \
 .option("checkpointLocation", "/ckpt/orders") \
 .start("s3://bucket/silver/orders/")

# Option B — write to a registered table
.writeStream \
 .option("checkpointLocation", "/ckpt/orders") \
 .toTable("silver.orders")
```

| | `.start(path)` | `.toTable(name)` |
|---|---|---|
| Registers in metastore | No — path only | Yes — table immediately queryable by name |
| Format required | Must set `.format("delta")` | Delta inferred from table definition |
| Unity Catalog support | Partial | Full |
| Schema from | Inferred at runtime | Table schema if exists, else inferred |

> Prefer `.toTable()` for production — it integrates with Unity Catalog, registers the table in the metastore, and is consistent with how batch writes work.

## Hands-On: Delta Streaming Source → foreachBatch MERGE

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stream_demo")
spark.sql("USE SCHEMA stream_demo")

# Source Delta table — new CDC rows land here (e.g., from Auto Loader)
spark.sql("""
  CREATE OR REPLACE TABLE bronze_events (
    op         STRING,
    event_id   STRING,
    user_id    STRING,
    amount     DOUBLE,
    updated_at TIMESTAMP
  ) USING DELTA
""")

# Silver target — the current state
spark.sql("""
  CREATE OR REPLACE TABLE silver_events (
    event_id   STRING,
    user_id    STRING,
    amount     DOUBLE,
    updated_at TIMESTAMP
  ) USING DELTA
""")

# Seed bronze with batch 1
spark.sql("""
  INSERT INTO bronze_events VALUES
    ('I', 'E001', 'U01', 100.0, '2024-01-15 09:00:00'),
    ('I', 'E002', 'U02', 200.0, '2024-01-15 09:01:00'),
    ('I', 'E003', 'U01',  50.0, '2024-01-15 09:02:00')
""")

print("Bronze seeded with 3 events")

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, desc

CKPT = "dbfs:/tmp/stream_demo_ckpt"
dbutils.fs.rm(CKPT, recurse=True)  # fresh start

def upsert_silver(batch_df, batch_id):
    """Apply CDC batch to silver_events via MERGE."""
    if batch_df.isEmpty():
        return

    # Deduplicate: keep latest change per event_id within the batch
    w = Window.partitionBy("event_id").orderBy(desc("updated_at"))
    deduped = (batch_df
               .withColumn("rn", row_number().over(w))
               .filter(col("rn") == 1)
               .drop("rn"))

    deduped.createOrReplaceTempView("cdc_batch")

    batch_df.sparkSession.sql("""
        MERGE INTO stream_demo.silver_events AS t
        USING cdc_batch                      AS s
        ON    t.event_id = s.event_id

        WHEN MATCHED AND s.op = 'D' THEN DELETE
        WHEN MATCHED               THEN UPDATE SET
          t.user_id    = s.user_id,
          t.amount     = s.amount,
          t.updated_at = s.updated_at
        WHEN NOT MATCHED           THEN INSERT
          (event_id, user_id, amount, updated_at)
          VALUES (s.event_id, s.user_id, s.amount, s.updated_at)
    """)
    print(f"Batch {batch_id}: merged {deduped.count()} records")


# Run stream — process what's in bronze now, then stop
q = (
    spark.readStream
         .format("delta")
         .table("stream_demo.bronze_events")
         .writeStream
         .foreachBatch(upsert_silver)
         .option("checkpointLocation", CKPT)
         .trigger(availableNow=True)
         .start()
)
q.awaitTermination()

print("\nSilver after run 1:")
spark.sql("SELECT * FROM stream_demo.silver_events ORDER BY event_id").show()

In [ ]:
# Simulate new CDC events arriving in bronze
spark.sql("""
  INSERT INTO bronze_events VALUES
    ('U', 'E001', 'U01', 150.0, '2024-01-15 10:00:00'),  -- update E001
    ('D', 'E003', 'U01',   0.0, '2024-01-15 10:01:00'),  -- delete E003
    ('I', 'E004', 'U03', 300.0, '2024-01-15 10:02:00')   -- new insert
""")

# Run 2 — picks up only the new rows (checkpoint skips batch 1)
q = (
    spark.readStream
         .format("delta")
         .table("stream_demo.bronze_events")
         .writeStream
         .foreachBatch(upsert_silver)
         .option("checkpointLocation", CKPT)
         .trigger(availableNow=True)
         .start()
)
q.awaitTermination()

print("\nSilver after run 2:")
spark.sql("SELECT * FROM stream_demo.silver_events ORDER BY event_id").show()
# Expected: E001 updated, E002 unchanged, E003 deleted, E004 inserted

## Managing Active Streaming Queries

In [ ]:
# Start a continuous stream (processingTime) for demonstration
continuous_q = (
    spark.readStream
         .format("delta")
         .table("stream_demo.bronze_events")
         .writeStream
         .format("memory")           # in-memory sink for demo
         .queryName("bronze_monitor")
         .outputMode("append")
         .trigger(processingTime="30 seconds")
         .start()
)

# List all active streaming queries in this session
print(f"Active streams: {len(spark.streams.active)}")
for q in spark.streams.active:
    print(f"  name={q.name}  id={q.id}  isActive={q.isActive}")

In [ ]:
# Inspect progress — lastProgress shows throughput, lag, state size
import json
progress = continuous_q.lastProgress
if progress:
    print(json.dumps({
        "id":           progress["id"],
        "batchId":      progress["batchId"],
        "numInputRows": progress["numInputRows"],
        "inputRowsPerSecond": progress["inputRowsPerSecond"],
    }, indent=2))
else:
    print("No progress yet — stream just started")

In [ ]:
# Stop a specific query by name
for q in spark.streams.active:
    if q.name == "bronze_monitor":
        q.stop()
        print(f"Stopped: {q.name}")

# Or stop all active streams
# for q in spark.streams.active:
#     q.stop()

In [ ]:
# Cleanup
dbutils.fs.rm(CKPT, recurse=True)
spark.sql("DROP SCHEMA IF EXISTS stream_demo CASCADE")

## Trigger Modes — Quick Reference

| Trigger | Code | Behaviour | Use case |
|---------|------|-----------|----------|
| Continuous (default) | no `.trigger()` | Micro-batch as fast as possible | Minimum latency |
| Fixed interval | `.trigger(processingTime="5 minutes")` | Batch every N | Controlled throughput |
| Available Now | `.trigger(availableNow=True)` | Process all pending, then stop | Scheduled batch-style jobs |
| Once *(deprecated)* | `.trigger(once=True)` | One micro-batch then stop | Use `availableNow` instead |

For Delta streaming in Databricks Jobs: **`availableNow=True`** is the standard pattern — it gives incremental processing with the cost profile of a batch job.

## Summary

- Any Delta table is a valid streaming source: `spark.readStream.format("delta").table("name")`. The stream delivers only new rows from each new commit.
- By default, updates and deletes in the source cause stream failure — use `ignoreChanges=true` to skip them (rows may be re-delivered), or use CDF for true change delivery.
- **`startingVersion`** and **`startingTimestamp`** control where a new stream begins. They are ignored if a checkpoint already exists.
- **Change Data Feed** (`readChangeFeed=true`) delivers insert, update (pre/post image), and delete records with `_change_type`, `_commit_version`, and `_commit_timestamp` system columns.
- **`foreachBatch` + MERGE** is the exam-critical pattern for streaming upserts — deduplicate the batch with `ROW_NUMBER`, register as a temp view, then run `MERGE INTO` via `batch_df.sparkSession.sql(...)`. `outputMode` is ignored with `foreachBatch`.
- **`.toTable(name)`** writes to a registered Delta table (preferred for production); **`.start(path)`** writes to an unregistered path.
- Manage active streams via `spark.streams.active`, `query.lastProgress`, `query.stop()`.
- Use **`availableNow=True`** trigger for scheduled Databricks Jobs — processes all pending data then exits cleanly.

---

**Section 3 — Incremental Data Processing — complete.** Next: Section 4 — Production Pipelines, starting with `12-delta-live-tables.ipynb`.